In [ ]:
import os, json,uuid
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

load_dotenv(override=True)

In [ ]:
# Initialization ->
load_dotenv(override=True)

groq_api_key=os.getenv('GROQ_API_KEY')
if groq_api_key:
        print(f"Groq api key is present and begins with the {groq_api_key[:8]}")
else:
        print(f"No API KEY found")
MODEL = "llama-3.3-70b-versatile"
openai = OpenAI()
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

In [ ]:
# prompt ->
system_message = """You are a concise travel assisstant.
Use tools when needed. Ask only one short follow-up question if required.
Return clear results and confirmations.
"""

In [ ]:
ticket_prices = {"delhi": 799, "banaras":899, "lucknow":1499, "kolkata":1200}
bookings = {}

In [ ]:
# now the function for the results ->
def search_trains(origin: str, destination:str, date:str):
        base = ticket_prices.get(destination.lower())
        return [
                {"train_id":f"{origin[:2].upper()}{destination[:2].upper()}-101","origin":origin,"destination":destination,"date":date,"price_inr":base},
                {"train_id":f"{origin[:2].upper()}{destination[:2].upper()}-202","origin":origin,"destination":destination,"date":date,"price_inr":base + 120},
        ]

In [ ]:
def get_ticket_price(destination_city:str):
        price = ticket_prices.get(destination_city.lower())
        return {"destination_city": destination_city, "price_inr":price if price is not None else None}

In [ ]:
def book_train(train_id:str,passenger_name:str):
        booking_id = str (uuid.uuid4())[:8]
        booking = {"booking_id":booking_id,"train_id":train_id,"passenger_name":passenger_name, "status":"CONFIRMED"}
        bookings[booking_id] = booking
        return booking

In [ ]:
def get_booking_status(booking_id:str):
        return bookings.get(booking_id,{"booking_id": booking_id,"status":"NOT_FOUND"})

In [ ]:
def cancel_booking(booking_id:str):
        b = bookings.get(booking_id)
        if not b:
                return {"booking_id":booking_id, "status":"NOT_FOUND"}
        b["status"] = "CANCELLED"
        return b

In [ ]:
# Now, calling the tools -> 
tools = [
        {
                "type":"function",
                "function":{
                        "name":"search_trains",
                        "description":"Search available trains for a given route and date.",
                        "parameters":{
                                "type":"object",
                                "properties":{
                                        "origin":{"type":"string"},
                                        "destination":{"type":"string"},
                                        "date":{"type":"string","description":"Date in YYYY-MM-DD"},
                                },
                                "required":["origin","destination","date"],
                                "additionalProperties":False,
                        },
                },
        },{
                "type":"function",
                "function":{
                        "name":"get_ticket_price",
                        "description":"Get a typical ticket price for a destination city.",
                        "parameters":{
                                "type":"object",
                                "properties":{
                                        "destination_city":{"type":"string"},
                                },
                                "required":["destination_city"],
                                "additionalProperties":False,
                        },
                },
        },{
                "type":"function",
                "function":{
                        "name":"book_train",
                        "description":"Book a train for a passenger using train_id.",
                        "parameters":{
                                "type":"object",
                                "properties":{
                                        "train_id":{"type":"string"},
                                        "passenger_name":{"type":"string"},
                                },
                                "required":["train_id","passenger_name"],
                                "additionalProperties":False,
                        },
                },
        },{
                "type":"function",
                "function":{
                        "name":"get_booking_status",
                        "description":"Get current status of an existing booking.",
                        "parameters":{
                                "type":"object",
                                "properties":{"booking_id":{"type":"string"}},
                                "required":["booking_id"],
                                "additionalProperties":False
                        },
                },
        },{
                "type":"function",
                "function":{
                        "name":"cancel_booking",
                        "description":"Cancel an existing booking.",
                        "parameters":{
                                "type":"object",
                                "properties":{"booking_id":{"type":"string"}},
                                "required":["booking_id"],
                                "additionalProperties":False
                        },
                },
        },
]

In [ ]:
tool_impl = {
        "search_trains":search_trains,
        "get_ticket_price": get_ticket_price,
        "book_train": book_train,
        "get_booking_status": get_booking_status,
        "cancel_booking": cancel_booking,
}

In [ ]:
def _clean_history(history):
        return [{"role":h["role"],"content": h["content"]} for h in history]

In [ ]:
def run_tool_call(tool_call):
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")
        result = tool_impl[fn_name](**args)
        return{
                "role":"tool",
                "tool_call_id": tool_call.id,
                "content":json.dumps(result)
        }

In [ ]:
def chat(message,history):
        history = _clean_history(history)
        messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
        
        response = groq_client.chat.completions.create(model=MODEL,messages=messages,tools=tools)
        
        while response.choices[0].finish_reason == "tool_calls":
                assisstant_message = response.choices[0].message
                messages.append({
    "role": "assistant",
    "content": assisstant_message.content or "",
    "tool_calls": [
        {
            "id": tc.id,
            "type": tc.type,
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments
            }
        }
        for tc in assisstant_message.tool_calls
    ]
})
                
                for toolCalls in assisstant_message.tool_calls:
                        messages.append(run_tool_call(toolCalls))
                
                response = groq_client.chat.completions.create(model=MODEL,messages=messages)
        return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat,title="Train Booking Assisstant (Tools + Function Calling)",examples=["Find me trains from delhi to banaras on 2026-05-12.",
                                                                                               "Book train Gatiman Express for Aditya Kumar Soni.",
                                                                                               "check booking status for booking is 987654.",
                                                                                               "Cancel booking for 987654",],).launch()